# 피처 엔지니어링

초기 100사이클 정보만 사용한다. 관측 총 사이클, 최종 용량, 라벨 상태, 수명은 입력 피처에서 제외한다.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
project = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project))
from src.features import load_cells, feature_matrix
from src.preprocess import DQ_FEATURES, policy_holdout, grouped_cv
cells = load_cells()
labelled = cells[cells.label_status == 'provided'].copy()


## 원본 추출과 ΔQ 창

`python -m src.preprocess --data-dir /원본/폴더`는 HDF5 원본의 제공 라벨과 초기 관측을 읽는다. ΔQ 창을 다른 사이클 창으로 대체하지 않는다.

In [2]:
import inspect
from src.preprocess import extract_cells
print(inspect.getsource(extract_cells))

def extract_cells(data_dir):
    """HDF5 원본의 summary·10/100사이클만 읽는다. 전체 cycles를 메모리에 올리지 않는다."""
    import h5py
    rows = []
    for batch_name, filename in BATCHES:
        with h5py.File(Path(data_dir) / filename, "r") as handle:
            batch = handle["batch"]
            for i in range(batch["cycle_life"].shape[0]):
                summary = handle[batch["summary"][i, 0]]
                cycles = handle[batch["cycles"][i, 0]]
                life = float(handle[batch["cycle_life"][i, 0]][()].reshape(-1)[0])
                if not np.isfinite(life) or life <= 0:
                    life = np.nan
                policy_data = handle[batch["policy_readable"][i, 0]][()].reshape(-1)
                policy = "".join(chr(int(v)) for v in policy_data)
                qd = _array(handle, summary["QDischarge"])
                cycle_numbers = _array(handle, summary["cycle"])
                vectors = []
                for number in (10, 100):
                    positions = np.flatnon

## 피처 변환 후보

`raw3`: ΔQ 분산·최솟값·평균. `logvar`: log10|분산|. `log3`: 세 통계량의 log10|값|. 수치 로그의 하한은 1e−15다. 후보별 변환 규칙은 고정하며, 로그 타깃 변환은 학습 데이터 안에서만 적용된다.

In [3]:
display(cells[['cell_id'] + DQ_FEATURES].head())
for feature_set in ['raw3','logvar','log3','full']:
    X = feature_matrix(labelled, feature_set)
    print(feature_set, X.shape, 'finite/nonmissing:', np.isfinite(X).sum())

,cell_id,dq_var,dq_min,dq_mean
0,Batch 1-0,0.000010,-0.008460,-0.002873
1,Batch 1-1,0.000010,-0.011004,-0.004100
2,Batch 1-2,0.000018,-0.017216,-0.004487
3,Batch 1-3,0.000036,-0.018961,-0.007456
4,Batch 1-4,0.000023,-0.013958,-0.005750


raw3 (119, 3) finite/nonmissing: 357
logvar (119, 1) finite/nonmissing: 119
log3 (119, 3) finite/nonmissing: 357
full (119, 8) finite/nonmissing: 952


## 정책 Hold-out과 학습 전처리

전처리 통계는 Hold-out 검증 셀에서 계산하지 않는다. 실제 튜닝에서는 전체 Pipeline이 각 CV fold에서 새로 fit된다.

In [4]:
from src.train import linear_net
train, valid = policy_holdout(labelled[labelled.batch == 'Batch 1'])
assert not set(train.charging_policy) & set(valid.charging_policy)
print('Train:', len(train), 'Valid:', len(valid))
display(pd.DataFrame({'train_policy': pd.Series(sorted(train.charging_policy.unique())), 'valid_policy': pd.Series(sorted(valid.charging_policy.unique()))}))
print(linear_net(log_target=True))

Train: 28 Valid: 8


,train_policy,valid_policy
0,5.4C(40%)-3.6C,4.4C(80%)-4.4C
1,5.4C(50%)-3C,4.8C(80%)-4.8C
2,5.4C(60%)-3.6C,6C(30%)-3.6C
3,5.4C(60%)-3C,7C(40%)-3.6C
4,5.4C(70%)-3C,8C(15%)-3.6C
5,5.4C(80%)-5.4C,NaN
6,6C(40%)-3.6C,NaN
7,6C(40%)-3C,NaN
8,6C(50%)-3.6C,NaN
9,6C(50%)-3C,NaN


GridSearchCV(error_score='raise',
             estimator=TransformedTargetRegressor(func=<ufunc 'log10'>,
                                                  inverse_func=<function inverse_log10 at 0x1168d6980>,
                                                  regressor=Pipeline(steps=[('simpleimputer',
                                                                             SimpleImputer(strategy='median')),
                                                                            ('standardscaler',
                                                                             StandardScaler()),
                                                                            ('elasticnet',
                                                                             ElasticNet(max_iter=20000))])),
             n_jobs=1,
             param_grid={'regressor__elasticnet__alpha': array([1.00000000e-04, 2.68269580e-04, 7.19685673e-04, 1.93069773e-03,
       5.17947468e-03, 1.38949549e-02, 3.7

## 로그 수명

원수명 MSE는 큰 수명 오차에 큰 가중치를 준다. 로그 수명 후보는 양수 예측을 만들고 상대적 오차를 다룬다. 내측 튜닝과 후보 선택은 MAPE를 사용한다. 로그 변환 자체가 배치 일반화를 보장하지는 않는다.